ベース：白地図 CARTO Light No Labels

国境線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin0.geojson

地域線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

人口ラスタ：WorldPop 'Syrian Arab Republic 100m Population 2026'
https://hub.worldpop.org/geodata/summary?id=75632
syr_worldpop_2026.tif

分析手法：Rasterio Resampling (Bilinear)

対象地域：
・Syria (National Scale)

その他：
・WorldPop人口ラスタを1/10サイズへ縮小
・Rasterio Resampling（Bilinear）により再サンプリング
・Transformを再計算し、地理的位置情報を維持
・Folium ImageOverlayで人口分布を可視化

In [1]:
# Core Libraries
# ラスタ処理・地図作成・可視化に必要なライブラリを読み込む

# Raster Processing
import rasterio
from rasterio.enums import Resampling

# Web Mapping
import folium

# Vector Processing
import geopandas as gpd

# Numerical Processing
import numpy as np

# Colour Mapping
import matplotlib.colors as mcolors

/Users/marisa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ETL Extract
# WorldPop人口ラスタを読み込む

pop_data = 'tif_1_syria_worldpop_2026.tif'

with rasterio.open(pop_data) as src:
    
    # ETL Transform
    # ラスタを1/10サイズへリサンプリングする
    upscale_factor = 0.1
    
    # 縮小後の行数・列数を計算する
    new_height = int(src.height * upscale_factor)
    new_width = int(src.width * upscale_factor)
    
    # Bilinear Resamplingで滑らかに縮小する
    data = src.read(
        1,
        out_shape=(new_height, new_width),
        resampling=Resampling.bilinear
    )
    
    # ETL Transform
    # リサンプリング後のTransformを再計算する
    # ラスタサイズ変更後も地理的位置がズレないよう維持する
    transform = src.transform * src.transform.scale(
        (src.width / data.shape[1]),
        (src.height / data.shape[0])
    )

# Check
# リサンプリング後のデータサイズを確認する
print("リサイズ後のデータ形状:", data.shape)

# Check
# Transformが再計算されているか確認する
print("新しく設計された位置情報（Transform）\n", transform)

リサイズ後のデータ形状: (601, 799)
新しく設計された位置情報（Transform）
 | 0.01, 0.00, 35.72|
| 0.00,-0.01, 37.32|
| 0.00, 0.00, 1.00|


In [3]:
# QGIS　オリジナルカラーコード

colors = [
    (0.00, "#7bb5a000"), 
    (0.20, "#7bb5a0ff"), 
    (0.40, "#2e9166ff"), 
    (0.60, "#226c4cff"), 
    (0.80, "#184d36ff"), 
    (1.00, "#0f3122ff")  
]
marisa_cmap = mcolors.LinearSegmentedColormap.from_list("Syria_Neon", colors)

In [4]:
# Map Setup
# 白地図ベースを作成

m = folium.Map(
    location=[34.8, 38.5], 
    zoom_start=7,
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a>'
)

In [5]:
# ETL Load
# 国境線を地図へ追加

folium.GeoJson('syr_admin0.geojson', name='Country',
               style_function=lambda x: {'color': 'black', 'weight': 3, 'fillOpacity': 0}).add_to(m)

# ETL Load
# Governorate境界線を地図へ追加

folium.GeoJson('syr_admin1.geojson', name='Governorate',
               style_function=lambda x: {'color': 'gray', 'weight': 1, 'fillOpacity': 0}).add_to(m)

In [6]:
# ETL Transform
# Folium表示用にNaNを置換する
# 欠損値を -1 に変換し、ImageOverlayで扱える配列に整える

data_clean = np.nan_to_num(data, nan=-1)

# Raster Bounds
# Transform情報からFolium用のバウンディングボックスを作成する
# ImageOverlayへ渡すため、南西（左下）と北東（右上）の座標を計算する

bounds = [
    [transform[5] + transform[4] * data.shape[0], transform[2]], # 南西（左下）
    [transform[5], transform[2] + transform[0] * data.shape[1]]  # 北東（右上）
]

# ETL Load
# リサンプリング後の人口ラスタをImageOverlayとして地図へ追加

folium.raster_layers.ImageOverlay(
    image=data_clean,
    bounds=bounds,
    colormap=marisa_cmap,
    opacity=0.8,
    name='Population Density (Resampled)'
).add_to(m)

In [7]:
# ETL Load
# 隣国ラベルを地図へ追加

neighbors = {
    "TÜRKIYE": [37.5, 37.5], 
    "IRAQ": [34.5, 42.0],         
    "JORDAN": [31.9, 36.5],       
    "LEBANON": [34.2, 35.0]       
}

for name, coords in neighbors.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="font-size: 14pt; font-weight: bold; color: gray; white-space: nowrap; text-align: center; width: 100px; margin-left: -50px;">{name}</div>'''
        )
    ).add_to(m)

In [8]:
# ETL Load
# Governorateラベルを地図へ追加

governorates = {
    "Aleppo": [36.2, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.8, 36.7],
    "Lattakia": [35.6, 36.1],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'<div style="font-size: 10px; color: black; font-weight: bold; white-space: nowrap; text-align: center; width: 100px; margin-left: -50px;">{name}</div>'
        )
    ).add_to(m)

In [9]:
# ETL Load
# HTMLマップとして保存

m.save('14_syria_raster_resample.html')